# R Master v9 Fix3 · Lap vs Mona Body Compare

**这版专门解决 iPhone Colab 断连：拆成短阶段 + Drive checkpoint + 每张图独立 Blender 进程。**

仍然是纯诊断：不改源模型、不接头、不转权重、不烘 Rest Pose、不导 VRM。

直接“全部运行”。如果手机中途断开，重新打开后再次“全部运行”，已经完成的 stage / PNG 会自动跳过。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import shutil, subprocess, zipfile, tarfile, json, os, hashlib

ROOT=Path("/content/drive/MyDrive/R_Master")
REF=ROOT/"reference"
CACHE=ROOT/"cache"
OUT=ROOT/"v9_body_compare"/"fix3_latest"
OUT.mkdir(parents=True,exist_ok=True)
LOCAL=Path("/content/r_master_v9_fix3")
LOCAL.mkdir(parents=True,exist_ok=True)

# Sources
mona_candidates=[
    ROOT/"v7c_refine/latest/R_Master_v7c_REFINED_PREVIEW.blend",
    ROOT/"v8c_head_beauty/latest/R_Master_v8c_HEAD_BEAUTY_PREVIEW.blend",
    ROOT/"v2/latest/R_Master_Align_v2_PREVIEW.blend",
]
MONA=next((p for p in mona_candidates if p.exists()),None)
LAPZIP=REF/"Lapine_Ver.1.11_A.zip"
if MONA is None: raise FileNotFoundError("Mona source not found")
if not LAPZIP.exists(): raise FileNotFoundError(LAPZIP)
print("✓ Mona:",MONA)
print("✓ Lapine:",LAPZIP)

# Reuse exact Blender archive cached by v8.
BLENDER_VERSION="4.4.3"
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
BLENDER=BDIR/"blender"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ Blender archive cache reused")
else:
    raise RuntimeError("v8 Blender cache missing; screenshot this line to Erdan")

if not BLENDER.exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
if not BLENDER.exists(): raise FileNotFoundError(BLENDER)
print("✓ Blender ready:",BLENDER)

# Extract Lapine BaseBody once and checkpoint it on Drive.
EXTRACTED=REF/"extracted"
EXTRACTED.mkdir(exist_ok=True)
LAPFBX=EXTRACTED/"Lapine_BaseBody.fbx"
if LAPFBX.exists() and LAPFBX.stat().st_size>10*1024*1024:
    print("✓ Lapine_BaseBody.fbx checkpoint reused")
else:
    TMP=LOCAL/"lap_extract"
    if TMP.exists(): shutil.rmtree(TMP)
    TMP.mkdir()
    with zipfile.ZipFile(LAPZIP,"r") as z:
        unity=[n for n in z.namelist() if n.lower().endswith("lapine.unitypackage")]
        if not unity: raise RuntimeError("Lapine.unitypackage missing")
        upkg=TMP/"Lapine.unitypackage"
        upkg.write_bytes(z.read(unity[0]))
    with tarfile.open(upkg,"r:*") as t:
        path_by_dir={}
        asset_by_dir={}
        for m in t.getmembers():
            parts=m.name.split("/")
            if len(parts)!=2: continue
            d,f=parts
            if f=="pathname":
                path_by_dir[d]=t.extractfile(m).read().decode("utf-8","ignore").strip()
            elif f=="asset":
                asset_by_dir[d]=m
        target=None
        for d,p in path_by_dir.items():
            if p.endswith("Assets/Models/FBX/Lapine_BaseBody.fbx") or p.endswith("Lapine_BaseBody.fbx"):
                target=d; break
        if target is None: raise RuntimeError("Lapine_BaseBody.fbx missing")
        LAPFBX.write_bytes(t.extractfile(asset_by_dir[target]).read())
    print("✓ Lapine_BaseBody.fbx extracted:",round(LAPFBX.stat().st_size/1024/1024,1),"MiB")

STAGE=OUT/"R_Master_v9_Fix3_COMPARE_STAGE.blend"
REPORT=OUT/"R_Master_v9_Fix3_report.json"
print("✓ setup complete")



In [ ]:
# Build comparison stage only. No rendering in this cell.
BUILD=LOCAL/"v9_fix3_build.py"
BUILD.write_text(r'''
import bpy,sys,os,json
from mathutils import Vector
argv=sys.argv[sys.argv.index("--")+1:]
def A(k):
    i=argv.index(k); return argv[i+1]
mona=A("--mona"); lap=A("--lap"); out=A("--out")
stage=os.path.join(out,"R_Master_v9_Fix3_COMPARE_STAGE.blend")
report_path=os.path.join(out,"R_Master_v9_Fix3_report.json")

print("[v9fix3] open Mona",flush=True)
bpy.ops.wm.open_mainfile(filepath=mona)
body=bpy.data.objects.get("R2_Mona_Main") or bpy.data.objects.get("Mona_Main")
rig=bpy.data.objects.get("R_Master_Align_v2_PREVIEW") or bpy.data.objects.get("Mona_Armature")
if not body: raise RuntimeError("Mona body not found")
body.name="CMP_Mona"

# Keep Mona body only among meshes to reduce memory.
for o in list(bpy.data.objects):
    if o.type=="MESH" and o!=body:
        bpy.data.objects.remove(o,do_unlink=True)
print("[v9fix3] Mona body verts",len(body.data.vertices),flush=True)

print("[v9fix3] import Lap BaseBody",flush=True)
bpy.ops.import_scene.fbx(filepath=lap)
imports=[o for o in bpy.context.scene.objects if o.type=="MESH" and o!=body]
if not imports: raise RuntimeError("Lap import produced no mesh")
lapbody=max(imports,key=lambda o:len(o.data.vertices))
lapbody.name="CMP_Lap"
for o in list(imports):
    if o!=lapbody: bpy.data.objects.remove(o,do_unlink=True)
print("[v9fix3] Lap body verts",len(lapbody.data.vertices),flush=True)

def pts(o): return [o.matrix_world@v.co for v in o.data.vertices]
def bounds(o):
    q=pts(o)
    mn=Vector((min(p.x for p in q),min(p.y for p in q),min(p.z for p in q)))
    mx=Vector((max(p.x for p in q),max(p.y for p in q),max(p.z for p in q)))
    return mn,mx

mmn,mmx=bounds(body); lmn,lmx=bounds(lapbody)
mh=mmx.z-mmn.z; lh=lmx.z-lmn.z
if mh<=0 or lh<=0: raise RuntimeError("bad bounds")
s=mh/lh
lapbody.scale*=s
bpy.context.view_layer.objects.active=lapbody
bpy.ops.object.transform_apply(location=False,rotation=False,scale=True)
lmn,lmx=bounds(lapbody)
mc=(mmn+mmx)*.5; lc=(lmn+lmx)*.5
lapbody.location += Vector((mc.x-lc.x,mc.y-lc.y,mmn.z-lmn.z))
bpy.context.view_layer.update()
lmn,lmx=bounds(lapbody)

def slab(o,zfrac,band=.012):
    mn,mx=bounds(o); z=mn.z+(mx.z-mn.z)*zfrac; tol=(mx.z-mn.z)*band
    q=[p for p in pts(o) if abs(p.z-z)<=tol]
    if not q:return None
    return {"width":max(p.x for p in q)-min(p.x for p in q),
            "depth":max(p.y for p in q)-min(p.y for p in q),
            "z":z}
levels={"shoulder":.80,"chest":.70,"waist":.58,"pelvis":.50,"upper_thigh":.43}
meas={k:{"mona":slab(body,v),"lap":slab(lapbody,v)} for k,v in levels.items()}

report={
 "ok":True,
 "stage":"R_Master_v9_Fix3_BodyCompare",
 "mona_source":mona,
 "lap_source":"Lapine_BaseBody.fbx",
 "mona_vertices":len(body.data.vertices),
 "lap_vertices":len(lapbody.data.vertices),
 "lap_height_scale_to_mona":s,
 "surface_slabs":meas,
 "source_modified":False,
 "weights_transferred":False,
 "rest_pose_baked":False,
 "final_vrm":False
}
with open(report_path,"w") as f: json.dump(report,f,indent=2)

bpy.ops.wm.save_as_mainfile(filepath=stage,check_existing=False)
print("[v9fix3] STAGE_OK",stage,flush=True)
''',encoding="utf-8")

need=not (STAGE.exists() and STAGE.stat().st_size>20*1024*1024 and REPORT.exists())
if need:
    LOG=OUT/"R_Master_v9_Fix3_build.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background","--python",str(BUILD),
         "--","--mona",str(MONA),"--lap",str(LAPFBX),"--out",str(OUT)]
    with LOG.open("w",encoding="utf-8") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            f.write(line)
            if "[v9fix3]" in line or "Error" in line or "Traceback" in line:
                print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(LOG.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError("v9 Fix3 stage build failed")
else:
    print("✓ compare stage checkpoint already exists")

print("✓ stage ready:",STAGE)
print(REPORT.read_text(encoding="utf-8")[:6000])



In [ ]:
# Prepare lightweight per-view renderer.
RENDER=LOCAL/"v9_fix3_render.py"
RENDER.write_text(r'''
import bpy,sys,os
from mathutils import Vector
argv=sys.argv[sys.argv.index("--")+1:]
def A(k):
    i=argv.index(k); return argv[i+1]
out=A("--out"); view=A("--view")
body=bpy.data.objects.get("CMP_Mona"); lap=bpy.data.objects.get("CMP_Lap")
if not body or not lap: raise RuntimeError("compare meshes missing")

# Neutral materials.
def mat(name,c,alpha=1):
    m=bpy.data.materials.get(name) or bpy.data.materials.new(name);m.use_nodes=True
    bs=m.node_tree.nodes.get("Principled BSDF")
    bs.inputs["Base Color"].default_value=(*c,1)
    bs.inputs["Roughness"].default_value=.86
    if "Specular IOR Level" in bs.inputs: bs.inputs["Specular IOR Level"].default_value=.08
    if alpha<1:
        bs.inputs["Alpha"].default_value=alpha
        try:m.surface_render_method="DITHERED"
        except:pass
    return m
body.data.materials.clear();body.data.materials.append(mat("CMP_Mona_Mat",(.38,.38,.40),1))
lap.data.materials.clear();lap.data.materials.append(mat("CMP_Lap_Mat",(.45,.66,.70),.48))
for o in (body,lap):
    for p in o.data.polygons:p.use_smooth=True

def bounds(o):
    q=[o.matrix_world@v.co for v in o.data.vertices]
    mn=Vector((min(p.x for p in q),min(p.y for p in q),min(p.z for p in q)))
    mx=Vector((max(p.x for p in q),max(p.y for p in q),max(p.z for p in q)))
    return mn,mx
mn,mx=bounds(body); c=(mn+mx)*.5; h=mx.z-mn.z; d=h*2.7

scene=bpy.context.scene
scene.render.engine="BLENDER_EEVEE_NEXT"
scene.render.resolution_x=512;scene.render.resolution_y=512;scene.render.resolution_percentage=100
scene.render.image_settings.file_format="PNG"
scene.world.color=(.025,.025,.03);scene.view_settings.exposure=-.8
for o in list(bpy.data.objects):
    if o.type=="ARMATURE": o.hide_render=True
    if o.type=="LIGHT": bpy.data.objects.remove(o,do_unlink=True)

def area(name,loc,en,size):
    dat=bpy.data.lights.new(name,"AREA");dat.energy=en;dat.size=size
    ob=bpy.data.objects.new(name,dat);scene.collection.objects.link(ob);ob.location=Vector(loc)
    ob.rotation_euler=(c-ob.location).to_track_quat("-Z","Y").to_euler()
area("Key",(c.x-h*.7,c.y-h*.9,c.z+h*.35),150,h*.9)
area("Fill",(c.x+h*.8,c.y-h*.3,c.z+h*.15),55,h*.8)

cd=bpy.data.cameras.new("cam");cam=bpy.data.objects.new("cam",cd);scene.collection.objects.link(cam);scene.camera=cam;cam.data.type="ORTHO"
spec={
 "front":(Vector((c.x,c.y-d,c.z)),c,h*1.10),
 "side":(Vector((c.x+d,c.y,c.z)),c,h*1.10),
 "back":(Vector((c.x,c.y+d,c.z)),c,h*1.10),
 "three_quarter":(Vector((c.x+d*.72,c.y-d*.72,c.z)),c,h*1.10),
 "chest_waist":(Vector((c.x,c.y-d,c.z+h*.17)),c+Vector((0,0,h*.17)),h*.42),
 "waist_pelvis_thigh":(Vector((c.x,c.y-d,c.z-h*.08)),c+Vector((0,0,-h*.08)),h*.46),
 "head_neck_cut":(Vector((c.x,c.y-d,c.z+h*.38)),c+Vector((0,0,h*.38)),h*.32),
}
pos,tgt,scale=spec[view]
cam.location=pos;cam.data.ortho_scale=scale;cam.rotation_euler=(tgt-pos).to_track_quat("-Z","Y").to_euler()
scene.render.filepath=os.path.join(out,f"R_Master_v9_Fix3_{view}.png")
bpy.ops.render.render(write_still=True)
print("[v9fix3] RENDER_OK",view,flush=True)
''',encoding="utf-8")
print("✓ renderer ready")



In [ ]:
# Render each view in a fresh Blender process. Every finished PNG is a Drive checkpoint.
views=["front","side","back","three_quarter","chest_waist","waist_pelvis_thigh","head_neck_cut"]
for i,v in enumerate(views,1):
    png=OUT/f"R_Master_v9_Fix3_{v}.png"
    if png.exists() and png.stat().st_size>15000:
        print(f"✓ [{i}/7] {v} checkpoint")
        continue
    print(f"[{i}/7] rendering {v}")
    r=subprocess.run(["xvfb-run","-a",str(BLENDER),"--background",str(STAGE),"--python",str(RENDER),
                      "--","--out",str(OUT),"--view",v],
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if r.returncode!=0:
        print(r.stdout[-12000:])
        raise RuntimeError(v+" render failed")
    print([x for x in r.stdout.splitlines() if "[v9fix3]" in x][-1:])
print("✓ all requested views checkpointed")



In [ ]:
from IPython.display import display,Image,Markdown
import zipfile
views=["front","side","back","three_quarter","chest_waist","waist_pelvis_thigh","head_neck_cut"]
for v in views:
    p=OUT/f"R_Master_v9_Fix3_{v}.png"
    if p.exists():
        display(Markdown("### "+v))
        display(Image(filename=str(p),width=460))

ZIP=OUT/"R_Master_v9_Fix3_Review.zip"
if ZIP.exists():ZIP.unlink()
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for v in views:
        p=OUT/f"R_Master_v9_Fix3_{v}.png"
        if p.exists():z.write(p,p.name)
    for f in ["R_Master_v9_Fix3_report.json","R_Master_v9_Fix3_build.log"]:
        p=OUT/f
        if p.exists():z.write(p,p.name)
print("✓ Review ZIP:",ZIP,round(ZIP.stat().st_size/1024/1024,2),"MiB")
from google.colab import files
files.download(str(ZIP))

